In [ ]:
# Celda 1 — Iniciar Spark y explorar schemas
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FlotaTransporte_L4") \
    .getOrCreate()

df_viajes = spark.read.csv("../data/raw/viajes.csv", header=True, inferSchema=True)
df_conductores = spark.read.parquet("../data/raw/conductores_final.parquet")

print("=== SCHEMA VIAJES ===")
df_viajes.printSchema()
print(f"Registros: {df_viajes.count()}\n")

print("=== SCHEMA CONDUCTORES ===")
df_conductores.printSchema()
print(f"Registros: {df_conductores.count()}")

=== SCHEMA VIAJES ===
root
 |-- id_viaje: integer (nullable = true)
 |-- id_conductor: integer (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- distancia_km: double (nullable = true)
 |-- duracion_min: integer (nullable = true)
 |-- tarifa: integer (nullable = true)
 |-- fecha: date (nullable = true)

Registros: 1000000

=== SCHEMA CONDUCTORES ===
root
 |-- id_conductor: long (nullable = true)
 |-- nombre: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- tipo_vehiculo: string (nullable = true)
 |-- zona: string (nullable = true)

Registros: 5000


# Celda 2 — Contexto de negocio (Markdown)
## Consultas Optimizadas en Spark SQL — Flota de Transporte

**Contexto de negocio:** Una empresa de transporte tipo ride-hailing necesita 
analizar 1M de viajes y 5K conductores para optimizar operaciones. Las consultas 
deben ser eficientes porque en producción operarían sobre volúmenes aún mayores.

**Objetivo:** Aplicar consultas avanzadas en Spark SQL y analizar cómo Catalyst 
y Tungsten optimizan la ejecución.

In [2]:
# Celda 3 — Exploración rápida de los datos

df_viajes.show(5)
df_conductores.show(5)

+--------+------------+-----------+------------+------------+------+----------+
|id_viaje|id_conductor|     ciudad|distancia_km|duracion_min|tarifa|     fecha|
+--------+------------+-----------+------------+------------+------+----------+
|  500001|           2| Valparaiso|       18.04|          64| 17807|2025-05-12|
|  500002|           3| Concepcion|       16.35|          50| 14964|2025-05-11|
|  500003|           4|  La Serena|        2.82|           8|  2932|2025-05-10|
|  500004|           5|Antofagasta|        2.99|          22|  3569|2025-05-09|
|  500005|           6|   Santiago|       44.89|         137| 39251|2025-05-08|
+--------+------------+-----------+------------+------------+------+----------+
only showing top 5 rows
+------------+-----------+------+-------------+--------+
|id_conductor|     nombre|rating|tipo_vehiculo|    zona|
+------------+-----------+------+-------------+--------+
|           1|Conductor_1|  4.19|          SUV|   Norte|
|           2|Conductor_2|  

Celda 4 — Markdown pregunta de negocio 1
### Consulta 1: Filtros y ordenamientos encadenados
**Pregunta de negocio:** ¿Cuáles son los 10 viajes más caros en Santiago 
con distancia superior a 20 km? Esto ayuda a identificar rutas premium 
para pricing dinámico.

In [3]:
# Celda 5 — Consulta 1 + explain
from pyspark.sql.functions import col

q1 = df_viajes \
    .filter((col("ciudad") == "Santiago") & (col("distancia_km") > 20)) \
    .orderBy(col("tarifa").desc()) \
    .limit(10)

q1.show()

+--------+------------+--------+------------+------------+------+----------+
|id_viaje|id_conductor|  ciudad|distancia_km|duracion_min|tarifa|     fecha|
+--------+------------+--------+------------+------------+------+----------+
|  625980|         981|Santiago|       45.99|         139| 41581|2026-03-19|
|  917790|        2791|Santiago|       45.98|         137| 41577|2025-09-25|
|  873020|        3021|Santiago|       45.95|         140| 41550|2025-05-23|
|  570770|         771|Santiago|       45.94|         141| 41548|2025-06-22|
|  552630|        2631|Santiago|       45.96|         134| 41545|2026-03-04|
|    6710|        1711|Santiago|       45.97|         143| 41534|2025-11-04|
|  856570|        1571|Santiago|       45.93|         140| 41528|2025-06-17|
|   42150|        2151|Santiago|       45.91|         143| 41509|2025-09-30|
|  698810|        3811|Santiago|       45.95|         131| 41507|2025-09-05|
|  837705|        2706|Santiago|        45.9|         132| 41501|2026-02-22|

In [4]:
# Celda 6 — Plan de ejecución consulta 1

q1.explain(True)

== Parsed Logical Plan ==
GlobalLimit 10
+- LocalLimit 10
   +- Sort [tarifa#22 DESC NULLS LAST], true
      +- Filter ((ciudad#19 = Santiago) AND (distancia_km#20 > cast(20 as double)))
         +- Relation [id_viaje#17,id_conductor#18,ciudad#19,distancia_km#20,duracion_min#21,tarifa#22,fecha#23] csv

== Analyzed Logical Plan ==
id_viaje: int, id_conductor: int, ciudad: string, distancia_km: double, duracion_min: int, tarifa: int, fecha: date
GlobalLimit 10
+- LocalLimit 10
   +- Sort [tarifa#22 DESC NULLS LAST], true
      +- Filter ((ciudad#19 = Santiago) AND (distancia_km#20 > cast(20 as double)))
         +- Relation [id_viaje#17,id_conductor#18,ciudad#19,distancia_km#20,duracion_min#21,tarifa#22,fecha#23] csv

== Optimized Logical Plan ==
GlobalLimit 10
+- LocalLimit 10
   +- Sort [tarifa#22 DESC NULLS LAST], true
      +- Filter ((isnotnull(ciudad#19) AND isnotnull(distancia_km#20)) AND ((ciudad#19 = Santiago) AND (distancia_km#20 > 20.0)))
         +- Relation [id_viaje#17,id_c

# 7
### Interpretación del plan de ejecución — Consulta 1

**¿Qué hizo Catalyst (optimizador lógico)?**
- Agregó verificaciones `isnotnull()` automáticamente en ciudad y distancia_km. 
  Esto evita procesar filas con valores nulos que nunca pasarían el filtro — 
  es trabajo que Catalyst ahorra sin que lo pidamos.
- Convirtió `cast(20 as double)` a la constante `20.0` directamente, eliminando 
  la conversión repetida fila por fila.

**¿Qué hizo el Physical Plan?**
- Usó `TakeOrderedAndProject` en vez de ordenar todo y luego cortar. Este 
  operador combina Sort + Limit en una sola operación: mantiene solo los 10 
  mayores en memoria sin ordenar el millón completo. 
  Analogía Lean: en vez de ordenar toda la bodega para encontrar los 10 
  productos más caros, revisás uno a uno y solo guardás los 10 mayores.
- `PushedFilters` muestra que los filtros se empujan al nivel de lectura del 
  archivo (FileScan). Spark descarta filas *mientras lee*, no después.

**Implicancia de negocio:** Esta consulta escalaría bien a 10M+ registros 
porque filtra temprano y nunca ordena el dataset completo.

# 8
### Consulta 2: Agregaciones con funciones estadísticas
**Pregunta de negocio:** ¿Cuál es el comportamiento de tarifas por ciudad? 
(promedio, desviación estándar, mínimo, máximo y total de viajes). 
Esto permite detectar ciudades con alta variabilidad en pricing.

In [5]:
# Consulta 
from pyspark.sql.functions import avg, stddev, min, max, count, round

q2 = df_viajes \
    .groupBy("ciudad") \
    .agg(
        round(avg("tarifa"), 0).alias("tarifa_promedio"),
        round(stddev("tarifa"), 0).alias("tarifa_stddev"),
        min("tarifa").alias("tarifa_min"),
        max("tarifa").alias("tarifa_max"),
        count("id_viaje").alias("total_viajes")
    ) \
    .orderBy(col("tarifa_promedio").desc())

q2.show()

+-----------+---------------+-------------+----------+----------+------------+
|     ciudad|tarifa_promedio|tarifa_stddev|tarifa_min|tarifa_max|total_viajes|
+-----------+---------------+-------------+----------+----------+------------+
|  La Serena|        21247.0|      11060.0|       880|     41576|      200000|
| Concepcion|        21239.0|      11079.0|       871|     41578|      200000|
|Antofagasta|        21230.0|      11063.0|       905|     41591|      200000|
|   Santiago|        21201.0|      11059.0|       899|     41581|      200000|
| Valparaiso|        21189.0|      11074.0|       887|     41562|      200000|
+-----------+---------------+-------------+----------+----------+------------+



In [6]:
q2.explain(True)



== Parsed Logical Plan ==
'Sort ['tarifa_promedio DESC NULLS LAST], true
+- Aggregate [ciudad#19], [ciudad#19, round(avg(tarifa#22), 0) AS tarifa_promedio#125, round(stddev(cast(tarifa#22 as double)), 0) AS tarifa_stddev#126, min(tarifa#22) AS tarifa_min#127, max(tarifa#22) AS tarifa_max#128, count(id_viaje#17) AS total_viajes#129L]
   +- Relation [id_viaje#17,id_conductor#18,ciudad#19,distancia_km#20,duracion_min#21,tarifa#22,fecha#23] csv

== Analyzed Logical Plan ==
ciudad: string, tarifa_promedio: double, tarifa_stddev: double, tarifa_min: int, tarifa_max: int, total_viajes: bigint
Sort [tarifa_promedio#125 DESC NULLS LAST], true
+- Aggregate [ciudad#19], [ciudad#19, round(avg(tarifa#22), 0) AS tarifa_promedio#125, round(stddev(cast(tarifa#22 as double)), 0) AS tarifa_stddev#126, min(tarifa#22) AS tarifa_min#127, max(tarifa#22) AS tarifa_max#128, count(id_viaje#17) AS total_viajes#129L]
   +- Relation [id_viaje#17,id_conductor#18,ciudad#19,distancia_km#20,duracion_min#21,tarifa#22,

### Interpretación del plan de ejecución — Consulta 2

**¿Qué hizo Catalyst (optimizador lógico)?**
- **Column pruning (poda de columnas):** En el Optimized Plan aparece un 
  `Project [id_viaje, ciudad, tarifa]` — Catalyst descartó las 4 columnas 
  que no participan en la consulta (id_conductor, distancia_km, duracion_min, 
  fecha). Menos datos leídos = menos memoria y I/O.
  Analogía Lean: no movés materiales que no se usan en esa estación de trabajo.

**¿Qué hizo el Physical Plan?**
- **HashAggregate en dos fases:**
  1. `partial_avg`, `partial_min`, `partial_max`, `partial_count`, 
     `partial_stddev` — cada partición calcula sus agregaciones locales.
  2. Un segundo `HashAggregate` combina los resultados parciales.
  Esto es clave: Spark NO envía 1M de filas por la red. Cada nodo resume 
  sus datos localmente y solo envía ~5 filas (una por ciudad) al nodo final.
  Analogía Lean: cada sucursal hace su propio reporte y solo envía el 
  resumen a la casa matriz, no cada boleta individual.

- **Exchange (Shuffle):** `hashpartitioning(ciudad, 200)` redistribuye los 
  datos para que todas las filas de una misma ciudad lleguen al mismo nodo. 
  Aparece un segundo Exchange para el ordenamiento final.

- **AdaptiveSparkPlan (AQE):** `isFinalPlan=false` indica que Spark puede 
  ajustar el plan en tiempo de ejecución según el volumen real de datos — 
  por ejemplo, reducir las 200 particiones si detecta que son pocas ciudades.

**Implicancia de negocio:** Con solo ~5 ciudades, el shuffle es mínimo. 
Si fueran 10.000 comunas, el costo de redistribución sería mayor y habría 
que considerar pre-particionamiento por zona geográfica.

### Consulta 3: Unión entre DataFrames (JOIN)
**Pregunta de negocio:** ¿Cuál es la tarifa promedio por tipo de vehículo 
y zona del conductor? Esto permite evaluar si ciertos tipos de vehículo 
generan más ingresos en zonas específicas para optimizar la asignación de flota.

In [7]:
q3 = df_viajes \
    .join(df_conductores, on="id_conductor", how="inner") \
    .groupBy("tipo_vehiculo", "zona") \
    .agg(
        round(avg("tarifa"), 0).alias("tarifa_promedio"),
        count("id_viaje").alias("total_viajes")
    ) \
    .orderBy(col("tarifa_promedio").desc())

q3.show(20)

+-------------+--------+---------------+------------+
|tipo_vehiculo|    zona|tarifa_promedio|total_viajes|
+-------------+--------+---------------+------------+
|       Pickup|Poniente|        21288.0|       50000|
|        Sedan|  Centro|        21274.0|       50000|
|    Hatchback| Oriente|        21267.0|       50000|
|        Sedan| Oriente|        21260.0|       50000|
|          SUV|Poniente|        21254.0|       50000|
|       Pickup| Oriente|        21253.0|       50000|
|    Hatchback|   Norte|        21252.0|       50000|
|       Pickup|  Centro|        21247.0|       50000|
|       Pickup|     Sur|        21236.0|       50000|
|        Sedan|Poniente|        21236.0|       50000|
|    Hatchback|  Centro|        21224.0|       50000|
|       Pickup|   Norte|        21217.0|       50000|
|    Hatchback|Poniente|        21211.0|       50000|
|    Hatchback|     Sur|        21181.0|       50000|
|          SUV|  Centro|        21176.0|       50000|
|          SUV|   Norte|    

In [8]:
q3.explain(True)

== Parsed Logical Plan ==
'Sort ['tarifa_promedio DESC NULLS LAST], true
+- Aggregate [tipo_vehiculo#27, zona#28], [tipo_vehiculo#27, zona#28, round(avg(tarifa#22), 0) AS tarifa_promedio#248, count(id_viaje#17) AS total_viajes#249L]
   +- Project [id_conductor#18, id_viaje#17, ciudad#19, distancia_km#20, duracion_min#21, tarifa#22, fecha#23, nombre#25, rating#26, tipo_vehiculo#27, zona#28]
      +- Join Inner, (cast(id_conductor#18 as bigint) = id_conductor#24L)
         :- Relation [id_viaje#17,id_conductor#18,ciudad#19,distancia_km#20,duracion_min#21,tarifa#22,fecha#23] csv
         +- Relation [id_conductor#24L,nombre#25,rating#26,tipo_vehiculo#27,zona#28] parquet

== Analyzed Logical Plan ==
tipo_vehiculo: string, zona: string, tarifa_promedio: double, total_viajes: bigint
Sort [tarifa_promedio#248 DESC NULLS LAST], true
+- Aggregate [tipo_vehiculo#27, zona#28], [tipo_vehiculo#27, zona#28, round(avg(tarifa#22), 0) AS tarifa_promedio#248, count(id_viaje#17) AS total_viajes#249L]
   

### Interpretación del plan de ejecución — Consulta 3

**¿Qué hizo Catalyst (optimizador lógico)?**
- **Column pruning en ambos lados del JOIN:**
  - Viajes: solo `id_viaje, id_conductor, tarifa` (3 de 7 columnas).
  - Conductores: solo `id_conductor, tipo_vehiculo, zona` (3 de 5 columnas).
  Catalyst descartó nombre, rating, ciudad, distancia, duración y fecha 
  porque no participan en la agregación final.
- **Filtros isnotnull automáticos:** agregó `isnotnull(id_conductor)` en 
  ambas tablas antes del join. Filas con conductor nulo nunca harían match, 
  así que se eliminan temprano.
- **Cast automático:** `id_conductor` es `int` en viajes y `long` en 
  conductores (Parquet). Catalyst insertó `cast(id_conductor as bigint)` 
  para compatibilizar los tipos.

**¿Qué hizo el Physical Plan?**
- **BroadcastHashJoin:** Spark detectó que conductores (5.000 filas) es 
  mucho más pequeña que viajes (1.000.000 filas), así que envió la tabla 
  completa de conductores a todos los nodos (`BroadcastExchange`). 
  Esto evita el shuffle costoso de la tabla grande.
  Analogía Lean: en vez de que cada estación de trabajo pida la lista de 
  conductores al sistema central, se imprime una copia para cada estación. 
  El costo de copiar 5K registros es mínimo vs. mover 1M de registros.
- **Parquet vs CSV:** Notar que conductores usa `Batched: true` (Parquet 
  permite lectura vectorizada) mientras viajes usa `Batched: false` (CSV 
  no lo permite). Parquet es más eficiente en lectura.

**Implicancia de negocio:** Si la tabla de conductores creciera a 1M+, 
Spark cambiaría automáticamente a SortMergeJoin (más lento pero viable). 
Mantener tablas de referencia pequeñas es una decisión de arquitectura 
que impacta directamente en performance.

### Consulta 4: Expresiones SQL con selectExpr
**Pregunta de negocio:** Clasificar cada viaje según su rentabilidad 
(tarifa por km) para identificar viajes de alta, media y baja eficiencia. 
Esto permite optimizar la asignación de conductores a rutas más rentables.

In [9]:
q4 = df_viajes \
    .selectExpr(
        "id_viaje",
        "ciudad",
        "distancia_km",
        "tarifa",
        "round(tarifa / distancia_km, 0) AS tarifa_por_km",
        """
        CASE 
            WHEN tarifa / distancia_km > 1500 THEN 'Alta'
            WHEN tarifa / distancia_km > 800  THEN 'Media'
            ELSE 'Baja'
        END AS eficiencia
        """
    )

q4.show(10)

+--------+-----------+------------+------+-------------+----------+
|id_viaje|     ciudad|distancia_km|tarifa|tarifa_por_km|eficiencia|
+--------+-----------+------------+------+-------------+----------+
|  500001| Valparaiso|       18.04| 17807|        987.0|     Media|
|  500002| Concepcion|       16.35| 14964|        915.0|     Media|
|  500003|  La Serena|        2.82|  2932|       1040.0|     Media|
|  500004|Antofagasta|        2.99|  3569|       1194.0|     Media|
|  500005|   Santiago|       44.89| 39251|        874.0|     Media|
|  500006| Valparaiso|        2.15|  4187|       1947.0|      Alta|
|  500007| Concepcion|        3.97|  4667|       1176.0|     Media|
|  500008|  La Serena|       20.91| 19437|        930.0|     Media|
|  500009|Antofagasta|       39.43| 35060|        889.0|     Media|
|  500010|   Santiago|       43.62| 37497|        860.0|     Media|
+--------+-----------+------------+------+-------------+----------+
only showing top 10 rows


In [10]:
q4.explain(True)

== Parsed Logical Plan ==
'Project ['id_viaje, 'ciudad, 'distancia_km, 'tarifa, 'round(('tarifa / 'distancia_km), 0) AS tarifa_por_km#294, CASE WHEN (('tarifa / 'distancia_km) > 1500) THEN Alta WHEN (('tarifa / 'distancia_km) > 800) THEN Media ELSE Baja END AS eficiencia#295]
+- Relation [id_viaje#17,id_conductor#18,ciudad#19,distancia_km#20,duracion_min#21,tarifa#22,fecha#23] csv

== Analyzed Logical Plan ==
id_viaje: int, ciudad: string, distancia_km: double, tarifa: int, tarifa_por_km: double, eficiencia: string
Project [id_viaje#17, ciudad#19, distancia_km#20, tarifa#22, round((cast(tarifa#22 as double) / cast(distancia_km#20 as double)), 0) AS tarifa_por_km#294, CASE WHEN ((cast(tarifa#22 as double) / cast(distancia_km#20 as double)) > cast(1500 as double)) THEN Alta WHEN ((cast(tarifa#22 as double) / cast(distancia_km#20 as double)) > cast(800 as double)) THEN Media ELSE Baja END AS eficiencia#295]
+- Relation [id_viaje#17,id_conductor#18,ciudad#19,distancia_km#20,duracion_min#21

### Interpretación del plan de ejecución — Consulta 4

**¿Qué hizo Catalyst (optimizador lógico)?**
- **Simplificación de constantes (Constant Folding):** Las comparaciones 
  `> 1500` y `> 800` se convirtieron a `> 1500.0` y `> 800.0` directamente. 
  Catalyst eliminó los `cast(1500 as double)` del Analyzed Plan para evitar 
  conversiones repetidas en cada fila.
- **Column pruning:** De 7 columnas solo lee 4: `id_viaje, ciudad, 
  distancia_km, tarifa`. Las columnas id_conductor, duracion_min y fecha 
  se descartan en la lectura.
- **`distancia_km` sin cast:** Como ya es double, Catalyst eliminó el 
  `cast(distancia_km as double)` redundante que aparecía en el Analyzed Plan.

**¿Qué hizo el Physical Plan?**
- **Sin shuffle ni Exchange:** Esta consulta es una transformación pura 
  fila por fila (Project). No hay agrupación ni join, así que cada partición 
  procesa sus filas independientemente sin comunicarse con otras.
  Es la operación más eficiente posible: lectura + cálculo, sin movimiento 
  de datos entre nodos.
- **selectExpr se ejecuta como Project:** Spark traduce las expresiones SQL 
  dentro de selectExpr al mismo operador que usaría con .select() + funciones. 
  No hay diferencia de performance entre ambas sintaxis.

**Implicancia de negocio:** Este tipo de clasificación (Alta/Media/Baja) es 
una lógica de negocio que escala linealmente — procesar 10M de viajes toma 
~10x más que 1M, sin costos adicionales de shuffle. Ideal para ETLs nocturnos 
que clasifiquen la operación del día.

## Resumen de optimizaciones de Catalyst y Tungsten

| Consulta | Optimización clave | Impacto |
|----------|-------------------|---------|
| Q1 - Filtros + Order | TakeOrderedAndProject | Evita ordenar 1M de filas; mantiene solo top 10 en memoria |
| Q2 - Agregaciones | HashAggregate parcial + final | Cada partición resume localmente; solo envía ~5 filas al nodo final |
| Q3 - JOIN | BroadcastHashJoin | Tabla chica (5K) se copia a cada nodo; evita shuffle de tabla grande (1M) |
| Q4 - selectExpr | Column pruning + Constant Folding | Lee solo 4 de 7 columnas; elimina conversiones redundantes |

**Patrón común:** En las 4 consultas, Catalyst aplicó **column pruning** 
automáticamente. Esto refuerza que Spark SQL optimiza independientemente 
de cómo escribimos la consulta — pero entender el plan nos permite validar 
que nuestras decisiones de diseño (formato de archivo, tamaño de tablas, 
estructura del join) favorecen las optimizaciones automáticas.

In [11]:
spark.stop()
print("Sesión Spark cerrada.")

Sesión Spark cerrada.
